# intro to HTS data
In this exercise we will cover the following:

 - File formats (FASTQ, SAM/BAM, VCF)
 - Mapping (single-end, paired-end) NGS data to a reference sequence
 - Read flags
 - VERY IMPORTANT, you need to identify the 'pipe' button on your computer '|'. That is the character that looks like a vertical bar, on a standard American keyboard this can be found by pressing shift+'button left of enter' or 'button above enter
 

In this exercise you will align a fastq file using bwa and generate a SAM file.

Due to the computational time we have created a reduced genome from one of the individuals from the wildebeest  project. The individual, CTauTzS_8872, has been sequenced using short read sequencing. For this exercise we have created a reduced reference genome.  

 The fastQ file CCTauTzS_8872.Goat.small.fq_1.gz has variable name with *_1.fq.gz  which is first read or the read pair.  
 
 
 ### running jupyter
 use Ctrl+ENTER to run code. 
 

In [ ]:
#############################################################
# ALL PATHS ARE SET HERE
# If the data or the software moves, this is the ONLY cell you
# need to change. No cell below this one uses a full path.
#############################################################

# where the shared data and software live
DATA=/course/data/NGSintro/animal
SOFTWARE=/course/data/NGSintro/software

# where you will do the exercise
WORK_DIR=$HOME/ngs_intro_animal

# input files
FASTQ_1=$DATA/CCTauTzS_8872.Goat.small.fq_1.gz
FASTQ_2=$DATA/CCTauTzS_8872.Goat.small.fq_2.gz
GOAT_REF=$DATA/goat.fa.gz

# programs
PICARD=$SOFTWARE/picard.jar
FASTQC=fastqc

# the same files as they are named inside WORK_DIR, and the sample name
# used for every file you create below
FQ1=$(basename $FASTQ_1)
FQ2=$(basename $FASTQ_2)
REF=$(basename $GOAT_REF)
SAMPLE=CTauTzS_8872

# the chromosome in the reduced reference
CHROM=NC_030808.1

# make WORK_DIR readable by the R cell further down
mkdir -p $WORK_DIR
echo $WORK_DIR > $HOME/.ngs_intro_animal_workdir

echo --programs that are installed:--
which samtools
which bwa
which angsd
which bcftools
which $FASTQC
ls $PICARD

echo; echo -Datasets that will be used-
echo ---pair of fastQ files
ls $FASTQ_1
ls $FASTQ_2
echo ---reference genome with index
ls $GOAT_REF*


First make a folder for the exercise and add symbolic links to the reference genomes and the fastQ files

In [ ]:
# enter the folder that was created in the first cell
cd $WORK_DIR

##make links to files and add them to the folder
# links to the two fastQ files
cp -sf  $FASTQ_1 .
cp -sf  $FASTQ_2 .
# link to reference genome
cp -sf  $GOAT_REF* .

echo --- files in folder ---
ls


In [ ]:
# the working directory was set in the first cell of the notebook
work_d <- readLines(path.expand("~/.ngs_intro_animal_workdir"))[1]
setwd(work_d)
getwd()


These are the files that you will be using. 

## Mapping one reduced genome
![ngs_files1](https://popgen.dk/albrecht/open/ngs_files1.png)
In this exercise you will align a fastq file using bwa and generate a SAM file.

Due to the computational time we have created a reduced genome from one of the wildebeest individuals. The individual, CCTauTzS_8872, has been sequenced using Illumina short-read sequencing. For this exercise we have created a reduced reference genome consisting only of chromosome NC_030808.1

The fastQ file CCTauTzS_8872.Goat.small.fq_1.gz has variable name with *_1.fq.gz, which is the first read of the read pair. The reference genome we will map to,goat.fa.gz, also only contains the chromosome NC_030808.1. 

Before we start mapping we want to perform some QC of the data.


 
# step 1: FastQ file and QC
![ngs_files2](https://popgen.dk/albrecht/open/ngs_files2.png)

### Viewing the input files (fastQ file)


view the fastq file (CTauTzS_8872_subset_R1.fastq.gz) using the head command and identify the reads and quality scores (ignore the Broken pipe warning)


In [ ]:
# -n determines the number of lines printed
gunzip -c $FQ1 | head -n 12


Identify the read names, the sequence, the separator line, and the base quality scores in the FASTQ output above.

**Questions**
 - Each read takes up four lines. Which line holds the bases, and which holds their quality scores?
 - The sequence line and the quality line are the same length. Why must that be true?

In [ ]:
# run to start quiz
from jupyterquiz import display_quiz
display_quiz('https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/ngs/quiz/fastq_format.json')



The below command count the number of lines in the file


In [ ]:
gunzip -c $FQ1 |  wc -l

 - How many lines do you have?
 - How many reads are in the data? (there are four lines per read)
 - Is the number of lines the same in the two FASTQ files? (modify the code above to check the other file)

 - How many lines do you have ????
 - How many Reads in the data ????
 - is the number of lines the same in the 2 fastQ files ???? ( modify the code above to see the number of lines in the other file)
 
 Take the quiz below. press enter after entering a number

In [ ]:
# run to start quiz
from jupyterquiz import display_quiz
display_quiz('https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/ngs/quiz/fastq_counts_animal.json')



#### reference fasta file

View the reference fasta file (goat.fa.gz) using the head command. You can modify the below uncommented code below to view other parts of the reference



In [ ]:
# first 20 lines
zcat $REF  | head -n 20

# last 1000 lines of the first million  lines (uncomment and modify below)
# gunzip -c $REF 2>/dev/null | head -n 1000000 | tail -n 1000

The reference contains capital and small letters. The small letters indicate a repeat region, which can be hard to map to.

**Questions**
 - The first line starts with `>`. What does that line tell you?
 - Why are there long stretches of `N` in a reference genome?
 - A FASTA file has no quality scores, while a FASTQ file does. Why does the reference not need them?

The reference contains capital and small letters. The small letter indicate that it is a repeat which can be hard to map to. 

#### fastqc

let's see if there are any issues with the sequencing reads

In [ ]:
 $FASTQC --nogroup $FQ1
 
 echo ---- fastQC has created this file ----
 ls *html

To view the report, switch to the main browser tab for the Jupyter notebook. Enter the folder `~/ngs_intro_animal/` and find the html file. Click the file to open the FastQC report.

![fastQC file](https://github.com/popgenDK/courses/blob/main/current_exercises/ngs/figures/fastqc_report_animal.png?raw=true)

**Questions**
 - How long are the reads?
 - Does the base quality drop towards the end of the reads? Why does that happen with Illumina sequencing?
 - Did any module fail or raise a warning?

### Run the cell below to take the quiz

In [ ]:
# run to start quiz
from jupyterquiz import display_quiz
display_quiz('https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/ngs/quiz/fastqc_animal.json')



# mapping / Aligning
![ngs_files3](https://popgen.dk/albrecht/open/ngs_files3.png)

- Mapping to a reference genome
- What is in the sam/bam file?
- Quality: Mapping vs Alignment
- Sequencing Depth


Align the reads using bwa. We use bwa in the exercises because it is fast and widely used. We first need to index the reference chromosome, followed by the actual aligning process. If should take around 1 min to finish. 


Once the index is made, the second step is to map the reads. There are several ways to do this, but I suggest you use the bwa mem mode, which is the most commonly used these days. Again you can run it with no arguments to get info about how to use it. 

In [ ]:
# see options
bwa mem

**Questions**
 - `bwa mem` printed its usage because you gave it no arguments. Which option sets the number of threads?
 - Which three inputs does `bwa mem` need in order to map paired-end data?

The number of options may be a bit overwhelming, but you can run it with no additional options, although I suggest you add "-t 5" to run 5 threads if your computer has multiple cores. It reads the compressed fastq files directly, so you need not decompress them. By default the result comes on stdout (in the terminal), so you have to redirect to a file, like the below command. 
We also want to add a read group name with information about where the reads comes from. This is very useful if you have sequencing data from multiple libraries.  
Now try to align the data


In [ ]:
# bwa command 
# bwa men -R readGroupName -t threads REF fastq_1 fast1_2

#align the data ( take ~ 1 min)
bwa mem -R '@RG\tID:foo\tSM:bar\tLB:library1' -t 5 $REF $FQ1 $FQ2 > $SAMPLE.sam

**Questions**
 - The output was sent to a file with `>`. What would have happened without it?
 - `-R '@RG\tID:foo\tSM:bar\tLB:library1'` adds a read group. Why does it matter to record which sample a read came from, once you start analysing many individuals together?

Wait until it is done - if there is no output it is still running and you will see [*]

Let's look at the generated sam file. The below command prints the first line of the file

In [ ]:


# view first 1 line of the sam file
samtools view $SAMPLE.sam 2>/dev/null  | head -n 1


You can read about the sam output here: https://bioinformatics-core-shared-training.github.io/cruk-summer-school-2017/Day1/Session5-alignedReads.html  

 - Identify the header and explain its contents. 
 - For the first read identify the following and fill in the (?????) below
     - the chromosome
     - the position of the first base of the read 
     - The mapping qualty
     - The alignment (cigar string)
     - the insert size (template length)
     - the read(the bases)
     - the base qualities



In [ ]:
# run to start quiz
from jupyterquiz import display_quiz
display_quiz('https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/ngs/quiz/sam_format.json')



 
Bonus information: To understand the flags (second column in the sam format) you can type a flag into this page and get the meaning: https://broadinstitute.github.io/picard/explain-flags.html
 


Let's try to find the number of reads  in the samfile.

In [ ]:
wc -l $SAMPLE.sam

**Questions**
 - Why is this not the same number as in the FASTQ file?
 - The count includes the header lines. How could you count only the alignment records?

- Why is it not the same number as in the fastQ file?



Fortunately there are tools to handle sam files, which will make your life easier. We will use the samtools program. First, you often need the compressed version of the sam format, which is called bam. You use samtools view for converting between formats. BAM files facilitates random access to genomic regions, but this requires the file to be sorted and requires  an index this is generated using the command below.
Converting sam to bam is done like this:

In [ ]:
#sam to bam
samtools view -b $SAMPLE.sam > $SAMPLE.bam

#sort bam file
samtools sort -o $SAMPLE.sorted.bam $SAMPLE.bam

#index bam file
samtools index $SAMPLE.sorted.bam

#see files and their sizes
echo --- files sizes ---
ls -lah $SAMPLE.sam $SAMPLE.bam $SAMPLE.sorted.bam

**Questions**
 - Compare the three file sizes printed above. Roughly how much smaller is the BAM than the SAM?
 - Is any information lost in the conversion?
 - Why did we have to sort the file before indexing it?

### Run the cell below to take the quiz

In [ ]:
# run to start quiz
from jupyterquiz import display_quiz
display_quiz('https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/ngs/quiz/sam_to_bam.json')

The bam file is a compressed version of sam, you can see it is about one-third of the sam file in size. 



We now have a functional alignment file that we can use for analysis. Let's first view the alignment at different part of the chromosome NC_030808.1. We will use tview to extract alignment. The option -d -w print 80 bases of the alignment to the terminal

In [ ]:
samtools tview  $SAMPLE.sorted.bam  -d T -w 80 -p $CHROM:130171


If the above alignment looks messy then reduce the printed bases. e.g. try -w 50

In the above the lines are

Line1: The position on chromosome NC_030808.1

Line2: The reference genome ( N if not provided)

Line3: The consensus sequence (If most or all reads have a G then the consensus is G)

Line4+:  (lines 4,5 ect) the reads alignment 


- When looking at the region starting with position NC_030808.1:130171 can you find a possible variable site?


Let's try to add the reference genome to make it easier to see the sequencing error and variable sites

In [ ]:
samtools tview $SAMPLE.sorted.bam  -d T -w 80 -p $CHROM:130161 $REF

 - can you find the site that is likely heterozygous?
 
 Some parts of the genome are hard to map to. Let's try another position
 - Change the position to NC_030808.1:156221. (modify above code a run)
 - How many likely variable sites can you see?
 - Are these sites heterozygous or is there another likely explanation?

 

 Another way to look at the genome is by generating a [pileup](http://samtools.sourceforge.net/samtools.shtml) format

In [ ]:
samtools mpileup $SAMPLE.sorted.bam -r $CHROM:130171-130191


Each line is a position with data containing CHR, position, Reference (N if not provided), Number of reads, bases at that position and their base quality scores. 
 - Can you find the heterozygous sites
 - When is this a format particularly useful?
 
 
 From the pileup it is easily to get the sequencing depth distribution

In [ ]:
samtools mpileup $SAMPLE.sorted.bam  | cut -f4 | sort -n | uniq -c >dep1

cat dep1

**Question**
 - The left column is the number of sites and the right one is the depth. Are positions with **zero** reads listed here? What does that mean for the numbers you are about to plot?


the left column is the number of sites and the right is the depth. 

View the distribution for this individuals using the following R command


In [ ]:

depth <- read.table("dep1")
d <- 1:15 #chosen depths to plot

barplot(depth[d+1,1],names=d,xlab="sequencing depth",ylab="Number of sites with sequencing depth ",col="mistyrose")


**Questions**
 - What is the most common sequencing depth?
 - At this depth, how confident would you be calling a heterozygous genotype from a single read?

### Run the cell below to take the quiz

In [ ]:
# run to start quiz
from jupyterquiz import display_quiz
display_quiz('https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/ngs/quiz/depth.json')

 - How do you think the depth will affect genotype and variant calling?
 



 
# Variant calling 


 ![ngs_files4](https://popgen.dk/albrecht/open/ngs_files4.png)

 - Pileup -> variant

### create VCF file
Let's create a VCF file for the first MB of CTauTzS_8872.sorted.bam. This is done using bcftools. The ploidy is diploid (2) for mammals but otherwise we use the default settings. However, before doing so we should remove duplicated reads ( read with the same starting points) as they are likely PCR duplicate

In [ ]:
## remove duplicates and index the new file
samtools rmdup -s $SAMPLE.sorted.bam $SAMPLE.md.bam
samtools index $SAMPLE.md.bam


## call variants
bcftools mpileup -Ou -f $REF $SAMPLE.md.bam -r $CHROM:1-1000000 | bcftools call --ploidy 2 -mv -a GQ  -Ov -o $SAMPLE.vcf

**Questions**
 - The VCF was made from `CTauTzS_8872.md.bam`, the file with duplicates removed, not from the sorted BAM. Why?
 - `--ploidy 2` was set. What would change for a haploid organism?

Let's have a look at the VCF file



In [ ]:
head -n 50 $SAMPLE.vcf 

**Questions**
 - The header lines start with `##`. What kind of information is stored there?
 - In the body, which columns hold the position, the reference allele and the alternative allele?
 - Can you find a site where the genotype is `0/1`? What does that mean?


 The header of the VCF contains meta information about what it in the file.
In the body of the file
 - Identify the position, the reference allele and the alternative allele of the file.
 - Identify the depth of each position
 - Identify the genotype quality for each genotype call
 
 There a many sites with too little information to call variants. Let apply some light filters. Here we remove sites with less than 8 reads and sites with a low quality score


In [ ]:
bcftools filter $SAMPLE.vcf -e 'QUAL<20 || DP < 8' > $SAMPLE.filt.vcf

head -n 100 $SAMPLE.filt.vcf 

**Questions**
 - The filter removed sites with `QUAL<20` or `DP<8`. Which of the two is doing most of the work on this low-depth data?
 - What is the risk of filtering too hard?

### Run the cell below to take the quiz

In [ ]:
# run to start quiz
from jupyterquiz import display_quiz
display_quiz('https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/ngs/quiz/vcf.json')


 
 
 - How many sites are called as variable?
 - Find a heterozygous site.  ( look for 0/1)


# Bonus exercise (Only do this part if you have finished the rest) 

 Look at the alignment for the position
 


In [ ]:
POS=6833

#the site you choose is the first base of the alignment so we center it by subtracting 50 to the position
POS50=$(($POS  - 50 ))
samtools tview  $SAMPLE.sorted.bam  -d T -w 100 -p $CHROM:$POS50



**Question**
 - Look at the column of bases at the centre of the alignment. Do all the reads agree?

We can use the mpileup option to get a summary of the data at that position

In [ ]:
  
echo -e "CHR\tPOS\tREF\tDEPTH\tBASES\tbaseQuality" > mpileup.file
samtools mpileup  $SAMPLE.sorted.bam -r $CHROM:$POS-$POS >> mpileup.file

echo --- pileup of the site which shows the bases and their score ---
column -t mpileup.file


 - At that position count the number of bases of different types
   - #A = ??
   - #C = ??
   - #G = ??
   - #T = ??

 - Calculate the genotype likelihoods and call the genotype for the site. 
 
You can use the shiny app to help with the calculations for calculating genotype likelihoods using the GATK model. Enter the BASES and their base qualities

https://popgen.dk/shiny/anders/GL/


The resulting values will not be exactly the same as in bcftools since bcftools uses a slightly different way of calculating the genotype likelihoods than the GATK model. 


 
 # Bonus exercise (Only do this part if you have finished the rest) 
 ## Bonus exercise 2 -  duplicated reads using Picardtools
 
 bwa actually fills in the mate information, but not all aligners do that, so we can run picard tools to fill in the mate information and sort the file according to position. We will output the file in the binary version of SAM which is BAM

In [ ]:
java -jar $PICARD FixMateInformation INPUT=$SAMPLE.sam \
OUTPUT=id.fixmate.srt.bam SORT_ORDER=coordinate

**Question**
 - Picard was given the SAM file and wrote a BAM file sorted by coordinate. Which two things did it change in one step?

View the header of the BAM file

In [ ]:
samtools view -H id.fixmate.srt.bam 

**Question**
 - Which lines describe the reference sequences, and which describe the programs that have touched the file?

picard didn't update the PG flag, so let us update the header information so that we have documented how we modified the file.

In [ ]:
(samtools view -H id.fixmate.srt.bam;echo -e "@PG\tID:fixmate\tPN:fixmate\tVN:2.60\tCL:stuff" ) >newhd
samtools reheader newhd id.fixmate.srt.bam > id.fixmate.srt2.bam



**Question**
 - What was added to the header by the command above, and why is it good practice to record it?

 - Validate that the header in file id.fixmate.srt2.bam  has been updated

In [ ]:
samtools view -H id.fixmate.srt2.bam 

**Question**
 - Compare this header with the one you printed before. Which line is new?

Now mark duplicates using picard

In [ ]:
java -jar $PICARD MarkDuplicates I=id.fixmate.srt2.bam \
O=id.fixmate.srt.md.bam  M=metrics;

 - Did picard update the PG flag of the header?
 - Did picard update anything else in the header?

NB you can view the header of a bamfile using 'samtools view -H'




In [ ]:
samtools view -H id.fixmate.srt.md.bam



## Bonus exercise 3 - clean you bam files using the FLAGS column

The second column in the SAM format is the very important FLAG. This will tell you about the state of the paired-end mapping, QC duplicates etc.


  
Using the samtools -F/-f you can discard/include flags that fulfill certain patterns. See http://broadinstitute.github.io/picard/explain-flags.html .

  1. How many reads have we marked as duplicate in the final file.
  2. How many properly mapped read pairs do we have? (Where both reads map to the same chr etc).
  3. How many mapped reads do we have ?
  4. How many unmapped reads do we have ?
  5. Find the distribution of the RNAMES of the unmapped reads!?

 Run the following command one at a time by uncommenting them

In [ ]:



#samtools view -f 1024 id.fixmate.srt.md.bam|wc -l
#samtools view -f 2 id.fixmate.srt.md.bam|wc -l
#samtools view -F 4 id.fixmate.srt.md.bam|wc -l
#samtools view -f 4 id.fixmate.srt.md.bam|wc -l
# samtools view -f 4 id.fixmate.srt.md.bam|cut -f3|sort -n |uniq -c




**Questions**
 - Uncomment the lines above one at a time and run them. How many reads are unmapped?
 - How many are marked as duplicates?
 - `-f 4` and `-F 4` give different counts. What is the relationship between the two numbers?

Compare with "samtools flagstat" command 


In [ ]:
samtools flagstat id.fixmate.srt.md.bam


**Questions**
 - What fraction of the reads mapped?
 - What fraction were properly paired?
 - Do the numbers agree with what you counted by hand with `-f` and `-F`?

### Run the cell below to take the quiz

In [ ]:
# run to start quiz
from jupyterquiz import display_quiz
display_quiz('https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/ngs/quiz/flags.json')

Make a new bamfile, where you keep only the reads where both ends map, and filter out those with a mapping quality below 10, and removing duplicates


In [ ]:
samtools view -f 2 -F 1024 id.fixmate.srt.md.bam -q 10 >new.bam

**Questions**
 - How many reads are left in `new.bam` compared with the file you started from?
 - Each of the three filters (`-f 2`, `-F 1024`, `-q 10`) removes a different kind of read. Which one removed the most?

---

**You have now been through the whole pipeline: FASTQ → SAM → BAM → VCF.**
For each of the three file formats, try to say in one sentence what it stores and what one line in it represents.